# M4 · Neural networks

**Outcome:** See why activations matter, then build and inspect an nn.Module.

Run cells with **Shift + Enter**. PyTorch is already installed in standard Colab runtimes.

## Experiment question

> What changes forward values and what controls backward slopes?

Before running code, write a prediction. Then observe the evidence, change one variable, and explain the difference.

In [ ]:
import torch
print('PyTorch', torch.__version__)
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')

## 1 · One neuron: weighted votes, bias, activation
Inspect every contribution before assembling whole layers.

In [ ]:
x = torch.tensor([1.5, -1.0])
w = torch.tensor([0.8, -0.4])
b = torch.tensor(0.2)
z = x @ w + b
output = torch.relu(z)
print('contributions:', x*w, 'z:', z.item(), 'output:', output.item())

## 2 · A linear stack still collapses
Verify that two affine transformations equal one affine transformation, then insert ReLU and compare.

In [ ]:
x = torch.linspace(-2, 2, 9)
w1, b1, w2, b2 = 1.4, 0.2, -1.1, 0.4
stacked = w2 * (w1*x + b1) + b2
collapsed = (w2*w1)*x + (w2*b1 + b2)
bent = w2 * torch.relu(w1*x + b1) + b2
print('linear stack matches:', torch.allclose(stacked, collapsed))
print('linear:', stacked)
print('with ReLU:', bent)

## 3 · Forward value and local slope
Use autograd to compare how much gradient survives at several inputs.

In [ ]:
for name, fn in [('ReLU', torch.relu), ('sigmoid', torch.sigmoid), ('tanh', torch.tanh)]:
    x = torch.tensor([-6., 0., 6.], requires_grad=True)
    y = fn(x)
    y.sum().backward()
    print(name, 'output:', y.detach().tolist(), 'slope:', x.grad.tolist())

## 4 · Package the layers

In [ ]:
class TinyNet(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = torch.nn.Sequential(
            torch.nn.Linear(2, 8), torch.nn.Tanh(), torch.nn.Linear(8, 1)
        )
    def forward(self, x):
        return self.layers(x).squeeze(-1)

model = TinyNet()
print(model)
print('parameters:', sum(p.numel() for p in model.parameters()))

## 5 · Logits belong with a stable loss
Verify that a shared offset changes neither probabilities nor cross-entropy.

In [ ]:
logits = torch.tensor([[1.2, -0.3], [101.2, 99.7]])
probs = logits.softmax(dim=1)
print(probs)
print('same probabilities:', torch.allclose(probs[0], probs[1]))
print('loss:', torch.nn.functional.cross_entropy(logits[:1], torch.tensor([0])).item())

## Reflection

1. What did you predict?
2. What evidence did the output provide?
3. Which one variable did you change?
4. How does the result connect to the lesson's mental model?